# 00 — Course Setup & WUFR-27 Reference State

This notebook establishes the common starting point for the suspension/vehicle-dynamics education series.

> **derive → calculate explicitly → call the shared model → correlate → interpret**

The notebooks are a teaching front end. Reusable physics, reviewed source data, and authority boundaries remain in the repository packages and source records.

## Setup goals

By the end of this notebook you should be able to load the reviewed WUFR-27 design-reference state without copying constants into the notebook, state the vehicle coordinate convention, identify the source records behind the basic vehicle parameters, and reproduce one transparent hand calculation.

There is intentionally no tire model, load-transfer solver, suspension solver, or widget here yet.

In [ ]:
from __future__ import annotations

from pathlib import Path
import math
import sys

from IPython.display import Markdown, display


def find_repo_root(start: Path | None = None) -> Path:
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'src').is_dir():
            return candidate
    raise FileNotFoundError('Could not find repository root from the current working directory')


# Allow the notebook to run directly from a repository checkout even when the
# active kernel/environment has not installed the package with `pip install -e .`.
ROOT = find_repo_root()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from pssd_vehicle import load_vehicle_reference


In [ ]:
SELECTOR = ROOT / 'configurations/education/WUFR27_EDUCATION_BASELINE_V0.toml'
vehicle = load_vehicle_reference(SELECTOR)

print(f'Repository:        {ROOT}')
print(f'Python executable: {sys.executable}')
print(f'Vehicle selector:  {vehicle.selector_id}')
print(f'Reference state:   {vehicle.state_id}')


## Coordinate convention

The reviewed whole-vehicle source uses a right-handed vehicle frame:

- **+x** forward
- **+y** vehicle left
- **+z** upward

Do not silently redefine axes or signs inside a lesson notebook. If a whiteboard derivation uses a temporary convention, state it explicitly and map back to the repository convention before calling shared code.

In [ ]:
rows = [
    ('Total mass', vehicle.total_mass_kg, 'kg'),
    ('Gravity', vehicle.g_mps2, 'm/s²'),
    ('Wheelbase', vehicle.geometry.wheelbase_m, 'm'),
    ('Front track', vehicle.geometry.front_track_m, 'm'),
    ('Rear track', vehicle.geometry.rear_track_m, 'm'),
    ('CG → front axle', vehicle.geometry.cg_to_front_axle_m, 'm'),
    ('CG → rear axle', vehicle.geometry.cg_to_rear_axle_m, 'm'),
    ('CG height above nominal road', vehicle.cg.height_above_nominal_road_m, 'm'),
]

table = [
    '| Quantity | Value | Unit |',
    '|---|---:|---|',
    *[f'| {name} | {value:.6f} | {unit} |' for name, value, unit in rows],
]
display(Markdown('\n'.join(table)))

print('Whole-vehicle adapter:', vehicle.whole_vehicle_adapter_id)
print('Static-gravity record:', vehicle.gravity_record_id)
print('Source configuration:', vehicle.source_configuration_id)


## First correlation pattern

Use the simplest possible example: total weight magnitude. On the whiteboard,

$$W = mg$$

Calculate it explicitly first. The shared object exposes the same derived quantity only as a convenience.

In [ ]:
weight_hand_N = vehicle.total_mass_kg * vehicle.g_mps2
weight_model_N = vehicle.total_weight_N

print(f'Hand calculation: {weight_hand_N:.3f} N')
print(f'Shared object:     {weight_model_N:.3f} N')
print(f'Residual:          {weight_model_N - weight_hand_N:.3e} N')

assert math.isclose(weight_hand_N, weight_model_N, rel_tol=0.0, abs_tol=1e-12)
assert vehicle.total_mass_kg > 0.0
assert vehicle.geometry.front_track_m > 0.0
assert vehicle.geometry.rear_track_m > 0.0
assert vehicle.cg.height_above_nominal_road_m > 0.0
assert math.isclose(
    vehicle.geometry.cg_to_front_axle_m + vehicle.geometry.cg_to_rear_axle_m,
    vehicle.geometry.wheelbase_m,
    rel_tol=0.0,
    abs_tol=1e-12,
)

print('Week 0 sanity checks passed.')


## What comes next

`01_contact_patch.ipynb` should begin from the physical question:

> **How can the chassis accelerate, brake, or turn if every external road force reaches it through four small tire contact patches?**

That lesson should derive the first force and moment relationships by hand before adding any meaningful solver to the shared codebase.